# Dataset Curation: ProteinBERT Embedding and Cluster-Based Purification

This notebook generates ProteinBERT global sequence embeddings for the combined positive/negative acetylation dataset, applies IncrementalPCA for dimensionality reduction, clusters with MiniBatchKMeans (k=2), and retains only sequences from pure clusters to reduce label noise.

**Input:** `positive_acetylation.csv`, `negative_acetylation.csv`  
**Output:** `distinct_positive_sequences.csv`, `distinct_negative_sequences.csv`

In [ ]:
!git clone https://github.com/nadavbra/protein_bert.git -q
!cd protein_bert && git submodule init && git submodule update && python setup.py install -q
import sys
sys.path.append('/content/protein_bert')

In [ ]:
import pandas as pd
import numpy as np
import h5py
import gc
from tqdm import tqdm
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

from proteinbert import load_pretrained_model
from proteinbert.conv_and_global_attention_model import get_model_with_hidden_layers_as_outputs

In [ ]:
model_seq_len = 32
raw_seq_len = 30
batch_size = 2
chunk_size = 5000
n_clusters = 2
embedding_file = 'embeddings.h5'
batch_pca_size = 10000
embedding_trim_dim = 512
n_pca_components = 50

In [ ]:
positive_df = pd.read_csv('positive_acetylation.csv')
negative_df = pd.read_csv('negative_acetylation.csv')
positive_df['label'] = 1
negative_df['label'] = 0
df = pd.concat([positive_df, negative_df], ignore_index=True)

print(f'Positive sequences: {len(positive_df)}')
print(f'Negative sequences: {len(negative_df)}')
print(f'Total: {len(df)}')

In [ ]:
def adjust_sequences(seqs, target_len):
    corrected = []
    for seq in seqs:
        if not isinstance(seq, str) or len(seq.strip()) == 0:
            continue
        seq = seq.strip()[:target_len]
        seq += 'X' * (target_len - len(seq))
        corrected.append(seq)
    return corrected

seqs = adjust_sequences(df['sequence'].tolist(), raw_seq_len)
df = df.iloc[:len(seqs)].copy()

print(f'Sequence lengths: {set(len(s) for s in seqs)}')
print(f'Total valid sequences: {len(seqs)}')

In [ ]:
print('Loading ProteinBERT...')
pretrained_model_generator, input_encoder = load_pretrained_model()
model = get_model_with_hidden_layers_as_outputs(
    pretrained_model_generator.create_model(model_seq_len)
)
print('Model loaded')

In [ ]:
print('Generating embeddings and streaming to disk...')
with h5py.File(embedding_file, 'w') as h5f:
    example = model.predict(input_encoder.encode_X([seqs[0]], raw_seq_len))[1]
    embedding_dim = example.shape[1]
    dset = h5f.create_dataset('embeddings', shape=(len(seqs), embedding_dim), dtype='float32')

    for i in tqdm(range(0, len(seqs), chunk_size)):
        batch_seqs = seqs[i:i + chunk_size]
        encoded = input_encoder.encode_X(batch_seqs, raw_seq_len)
        _, global_repr = model.predict(encoded, batch_size=batch_size, verbose=0)
        dset[i:i + len(batch_seqs)] = global_repr
        del encoded, global_repr
        gc.collect()

print(f'Embeddings saved: {embedding_file}')

In [ ]:
with h5py.File(embedding_file, 'r') as h5f:
    dset = h5f['embeddings'][:, :embedding_trim_dim]
    n = len(dset)
    mean = np.zeros(embedding_trim_dim, dtype=np.float64)
    total = 0
    for i in tqdm(range(0, n, batch_pca_size)):
        batch = dset[i:i + batch_pca_size]
        mean += batch.sum(axis=0)
        total += len(batch)
    mean /= total

    std = np.zeros(embedding_trim_dim, dtype=np.float64)
    for i in range(0, n, batch_pca_size):
        batch = dset[i:i + batch_pca_size]
        std += ((batch - mean) ** 2).sum(axis=0)
    std = np.sqrt(std / total)
    std[std == 0] = 1.0

print('Mean/std computed')

In [ ]:
ipca = IncrementalPCA(n_components=n_pca_components, batch_size=batch_pca_size)
with h5py.File(embedding_file, 'r') as h5f:
    dset = h5f['embeddings'][:, :embedding_trim_dim]
    for i in tqdm(range(0, len(dset), batch_pca_size), desc='Fitting PCA'):
        batch = (dset[i:i + batch_pca_size] - mean) / std
        ipca.partial_fit(batch)

X_pca = []
with h5py.File(embedding_file, 'r') as h5f:
    dset = h5f['embeddings'][:, :embedding_trim_dim]
    for i in tqdm(range(0, len(dset), batch_pca_size), desc='Transforming'):
        batch = (dset[i:i + batch_pca_size] - mean) / std
        X_pca.append(ipca.transform(batch))

X_pca = np.vstack(X_pca)
np.save('X_pca.npy', X_pca)
df.to_csv('df_with_labels.csv', index=False)
print(f'PCA shape: {X_pca.shape}')

In [ ]:
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1000)
df['cluster'] = kmeans.fit_predict(X_pca)

cluster_table = pd.crosstab(df['cluster'], df['label'])
print('Cluster composition:')
print(cluster_table)

In [ ]:
positive_filtered = df[(df['cluster'] == 0) & (df['label'] == 1)]
negative_filtered = df[(df['cluster'] == 1) & (df['label'] == 0)]

positive_filtered.to_csv('distinct_positive_sequences.csv', index=False)
negative_filtered.to_csv('distinct_negative_sequences.csv', index=False)

print(f'Saved {len(positive_filtered)} positive sequences')
print(f'Saved {len(negative_filtered)} negative sequences')

In [ ]:
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_pca)
df['pc1'] = X_2d[:, 0]
df['pc2'] = X_2d[:, 1]
df['label_name'] = df['label'].map({1: 'Positive', 0: 'Negative'})
df['cluster_name'] = df['cluster'].map({0: 'Cluster A', 1: 'Cluster B'})

sample = df.sample(50000, random_state=42)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample, x='pc1', y='pc2',
                hue='cluster_name', style='label_name',
                palette='Set1', s=10, alpha=0.6)
plt.title('Sequence clusters in PCA space')
plt.tight_layout()
plt.savefig('cluster_visualization.png', dpi=200)
plt.show()